# Exercise Analytics Engineer

## 1. Tasks

Imagine you are working as an Analytics Engineer for a company offering a
mobile app. This app collects millions of data points from users every day. Your
goal is to transform and organize this data so that not only you can derive
valuable insights from it, but also product managers and marketers can easily
query it.
Attached to this exercise you can find an SQLite database containing two tables.
In the following, we give a brief description of the data.
This first table (events) contains 43,479 telemetric events for one week from a
hypothetical mobile app. All users contained in the database are new users. Each
row in this table describes a single event by a user and contains the following
columns:  

● user_id  

● event_name  

● event_timestamp (unix timestamp measured in microseconds)  

● platform  

● os (= operating system)  

● country  

● ad_revenue (only set for particular events related to monetization)  

● tracker_name (the name of the campaign if a user was acquired via paid
marketing or “Unattributed“ if this is an organic user.  

The second table (user_acquisition) contains information about user
acquisition campaigns. These campaigns were run to acquire new users via
different ad networks. The data contains for each day and campaign a tracker
name (i.e., the name of the campaign) and the amount spent on this day for this
campaign. To be more precise, the table contains the following columns:  

● date  

● tracker_name  

● costs  

Please answer the following questions and implement the necessary tasks in a
programming language of your choice (hint: the Python standard library offers a
module sqlite3 which can be easily used for the task to read a file based database. 
Of course, you are also free to choose other packages or programming
languages).

1. What’s the total number of users present in the dataset?
2. List the number of installs per country.
3. In this exercise, you will calculate the retention for a specific cohort.
○ How many users installed the app on August 2, 2022 in Germany on
Android?
○ How many of these users are active on the first, third, and fourteenth
day after the install respectively? (I.e., count users for all three days
separately)
○ How much are those in percent? These are called day 1, day 3, and
day 14 retention.
4. Create a view named marketing that provides the following columns per
day and per campaign:
○ day
○ tracker_name
○ number_of_installs
○ costs (costs spent on this day for the specific campaign)
○ total_revenue (revenue from the users acquired on this day from
this campaign. Make use of the column ad_revenue from events)
5. Query the view marketing and report the Costs per Install (CPI) on August
6, 2022, for campaign “google_campaign1”?

Final Remarks  

● Please submit your documented code along with instructions on how to
run the code and the answers to the questions above.  

● Please only spend 120 minutes on the exercise. We know that it is
challenging to complete all tasks but please respect the time limit.

Attachment
- exercise.db.zip

## 2. Ideas

Based on the provided task list  I would approach the task using **Python (Jupyter Notebook)** and the **sqlite3** module step by step in the following manner:

---

## 🧰 Setup: Load and Explore the Database

```python
import sqlite3
import pandas as pd

# Connect to the SQLite database
conn = sqlite3.connect("exercise.db")

# Preview tables
pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn)
```

---

## 1️⃣ Total Number of Users

```python
query = "SELECT COUNT(DISTINCT user_id) AS total_users FROM events;"
pd.read_sql(query, conn)
```

---

## 2️⃣ Number of Installs per Country

Assuming "install" is represented by a specific `event_name` (e.g., `"install"` or similar):

```python
query = """
SELECT country, COUNT(DISTINCT user_id) AS installs
FROM events
WHERE event_name = 'install'
GROUP BY country;
"""
pd.read_sql(query, conn)
```

---

## 3️⃣ Retention for Cohort: Germany, Android, August 2, 2022

### a. Users who installed on 2022-08-02

```python
from datetime import datetime, timedelta

# Convert date to microseconds
install_day = datetime(2022, 8, 2)
start_ts = int(install_day.timestamp() * 1_000_000)
end_ts = int((install_day + timedelta(days=1)).timestamp() * 1_000_000)

query = f"""
/* Select Germany, Android, August 2, 2022 */
SELECT DISTINCT user_id
FROM events
WHERE event_name = 'install'
AND country = 'Germany'
AND platform = 'Android'
AND event_timestamp BETWEEN {start_ts} AND {end_ts};
"""
cohort_users = pd.read_sql(query, conn)
```

### b. Retention on Day 1, 3, 14

```python
def retention_day(day_offset):
    day_start = install_day + timedelta(days=day_offset)
    ts_start = int(day_start.timestamp() * 1_000_000)
    ts_end = int((day_start + timedelta(days=1)).timestamp() * 1_000_000)

    query = f"""
    SELECT DISTINCT user_id
    FROM events
    WHERE event_timestamp BETWEEN {ts_start} AND {ts_end}
    AND user_id IN ({','.join(map(str, cohort_users['user_id']))});
    """
    return pd.read_sql(query, conn)

day1 = retention_day(1)
day3 = retention_day(3)
day14 = retention_day(14)

# Retention percentages
total = len(cohort_users)
retention = {
    "Day 1": len(day1) / total * 100,
    "Day 3": len(day3) / total * 100,
    "Day 14": len(day14) / total * 100
}
```

---

## 4️⃣ Create View `marketing`

A view in SQL is a virtual table which does not store data itself, but instead presents the result of a stored query as if it were a table. We can query it just like a regular table, however in reality, it dynamically pulls data from the underlying tables.

```python
query = """
-- Create a view
CREATE VIEW IF NOT EXISTS marketing AS
SELECT
    ua.date AS day,
    ua.tracker_name,
    COUNT(DISTINCT e.user_id) AS number_of_installs,
    ua.costs,
    SUM(e.ad_revenue) AS total_revenue
FROM user_acquisition ua
LEFT JOIN events e
    ON e.tracker_name = ua.tracker_name
    AND DATE(e.event_timestamp / 1000000, 'unixepoch') = ua.date
    AND e.event_name = 'install'
GROUP BY ua.date, ua.tracker_name;
"""
conn.execute(query)
```

---

## 5️⃣ Query CPI for August 6, 2022, "google_campaign1"

```python
query = """
/* Select August 6, 2022 */
SELECT
    costs,
    number_of_installs,
    ROUND(costs * 1.0 / number_of_installs, 2) AS CPI
FROM marketing
WHERE day = '2022-08-06' AND tracker_name = 'google_campaign1';
"""
pd.read_sql(query, conn)
```

---

## 🧾 Final Notes

- We can wrap each step in Markdown cells to document our logic.
- We should use `conn.close()` at the end to cleanly close the connection.
- If needed, we can export results using `df.to_csv()` or `df.to_excel()`.

---

This is a clean list of `!pip install` commands we can run in our Jupyter Notebook to cover everything needed for the **Analytics Engineer Coding Exercise** using **Python** and **SQLite**:

```python
# Core packages
!pip install pandas
!pip install matplotlib  # Optional: for visualizing retention or CPI trends

# For file watching or hot reloads (optional)
!pip install watchdog

# If you plan to use SQL magic in Jupyter
!pip install ipython-sql
```

There is no need to install `sqlite3` — it is a part of Python’s standard library.

## 3. Notebook scaffolding

Here's a clean, well-documented Python Jupyter Notebook scaffold that answers all questions from the **Analytics Engineer Coding Exercise** using `sqlite3` and `pandas`:

---

## 📘 Analytics Engineer Coding Exercise — Jupyter Notebook

### 🔧 Setup

```python
# Install necessary packages (run once)
!pip install pandas matplotlib ipython-sql
```

```python
# Import libraries
import sqlite3
import pandas as pd
from datetime import datetime, timedelta
```

```python
# Connect to SQLite database
conn = sqlite3.connect("exercise.db")  # Replace with actual path if needed
```

---

### 📋 1. Total Number of Users

```python
query = "SELECT COUNT(DISTINCT user_id) AS total_users FROM events;"
pd.read_sql(query, conn)
```

---

### 🌍 2. Number of Installs per Country

```python
query = """
SELECT country, COUNT(DISTINCT user_id) AS installs
FROM events
WHERE event_name = 'install'
GROUP BY country;
"""
pd.read_sql(query, conn)
```

---

### 📈 3. Retention for Germany / Android / 2022-08-02

#### a. Define cohort

```python
install_day = datetime(2022, 8, 2)
start_ts = int(install_day.timestamp() * 1_000_000)
end_ts = int((install_day + timedelta(days=1)).timestamp() * 1_000_000)

query = f"""
SELECT DISTINCT user_id
FROM events
WHERE event_name = 'install'
AND country = 'Germany'
AND platform = 'Android'
AND event_timestamp BETWEEN {start_ts} AND {end_ts};
"""
cohort_users = pd.read_sql(query, conn)
cohort_users.head()
```

#### b. Retention function

```python
def retention_day(day_offset):
    day_start = install_day + timedelta(days=day_offset)
    ts_start = int(day_start.timestamp() * 1_000_000)
    ts_end = int((day_start + timedelta(days=1)).timestamp() * 1_000_000)

    query = f"""
    SELECT DISTINCT user_id
    FROM events
    WHERE event_timestamp BETWEEN {ts_start} AND {ts_end}
    AND user_id IN ({','.join(map(str, cohort_users['user_id']))});
    """
    return pd.read_sql(query, conn)
```

#### c. Retention results

```python
day1 = retention_day(1)
day3 = retention_day(3)
day14 = retention_day(14)

total = len(cohort_users)
retention = {
    "Day 1": round(len(day1) / total * 100, 2),
    "Day 3": round(len(day3) / total * 100, 2),
    "Day 14": round(len(day14) / total * 100, 2)
}
retention
```

---

### 🧮 4. Create View `marketing`

```python
query = """
CREATE VIEW IF NOT EXISTS marketing AS
SELECT
    ua.date AS day,
    ua.tracker_name,
    COUNT(DISTINCT e.user_id) AS number_of_installs,
    ua.costs,
    SUM(e.ad_revenue) AS total_revenue
FROM user_acquisition ua
LEFT JOIN events e
    ON e.tracker_name = ua.tracker_name
    AND DATE(e.event_timestamp / 1000000, 'unixepoch') = ua.date
    AND e.event_name = 'install'
GROUP BY ua.date, ua.tracker_name;
"""
conn.execute(query)
```

---

### 💰 5. CPI for 2022-08-06 and `google_campaign1`

```python
query = """
SELECT
    costs,
    number_of_installs,
    ROUND(costs * 1.0 / number_of_installs, 2) AS CPI
FROM marketing
WHERE day = '2022-08-06' AND tracker_name = 'google_campaign1';
"""
pd.read_sql(query, conn)
```

---

### ✅ Wrap-up

```python
# Close connection
conn.close()
```

---

## 4. Plotting (Visualization)

Now we can extend our Jupyter Notebook to include **visual dashboards** and **PDF export** of the results. This will make our analysis both insightful and presentation-ready.

---

## 📊 1. Retention Curve Visualization

```python
import matplotlib.pyplot as plt

# Retention dictionary from earlier
retention_days = ["Day 1", "Day 3", "Day 14"]
retention_values = [retention["Day 1"], retention["Day 3"], retention["Day 14"]]

plt.figure(figsize=(8, 5))
plt.plot(retention_days, retention_values, marker='o', linestyle='-', color='teal')
plt.title("Retention Curve – Germany / Android / 2022-08-02")
plt.xlabel("Day")
plt.ylabel("Retention (%)")
plt.grid(True)
plt.ylim(0, 100)
plt.show()
```

---

## 📈 2. CPI Trend Visualization

```python
# Load CPI data from marketing view
query = """
SELECT day, tracker_name, costs, number_of_installs,
       ROUND(costs * 1.0 / number_of_installs, 2) AS CPI
FROM marketing
WHERE tracker_name = 'google_campaign1'
ORDER BY day;
"""
cpi_df = pd.read_sql(query, conn)

# Plot CPI over time
plt.figure(figsize=(10, 5))
plt.plot(cpi_df["day"], cpi_df["CPI"], marker='o', color='darkorange')
plt.title("CPI Trend – google_campaign1")
plt.xlabel("Date")
plt.ylabel("Cost Per Install (CPI)")
plt.xticks(rotation=45)
plt.grid(True)
plt.tight_layout()
plt.show()
```

---

## 🧾 3. Export Results to PDF

We can use `matplotlib.backends.backend_pdf` to export plots and tables:

```python
from matplotlib.backends.backend_pdf import PdfPages

with PdfPages("analytics_dashboard.pdf") as pdf:
    # Retention plot
    plt.figure(figsize=(8, 5))
    plt.plot(retention_days, retention_values, marker='o', color='teal')
    plt.title("Retention Curve – Germany / Android / 2022-08-02")
    plt.xlabel("Day")
    plt.ylabel("Retention (%)")
    plt.grid(True)
    plt.ylim(0, 100)
    pdf.savefig()
    plt.close()

    # CPI plot
    plt.figure(figsize=(10, 5))
    plt.plot(cpi_df["day"], cpi_df["CPI"], marker='o', color='darkorange')
    plt.title("CPI Trend – google_campaign1")
    plt.xlabel("Date")
    plt.ylabel("Cost Per Install (CPI)")
    plt.xticks(rotation=45)
    plt.grid(True)
    plt.tight_layout()
    pdf.savefig()
    plt.close()

    # Optional: Export table as image
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.axis('tight')
    ax.axis('off')
    table = ax.table(cellText=cpi_df.values, colLabels=cpi_df.columns, loc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(8)
    pdf.savefig()
    plt.close()
```

---

## ✅ Conclusion

- We now have a PDF file `analytics_dashboard.pdf` with retention and CPI visuals.

---

## 5. Ideas for future investigation

The datasets in the **Analytics Engineer Coding Exercise** open up a rich field of exploratory and causal analysis. Beyond the core metrics like installs, retention, and CPI, we could flesh out a deeper set of **investigable questions and correlations**, grouped by theme:

---

## 📊 User Behavior & Retention

### 🔍 Questions
- What is the average session frequency per user by country or platform?
- How does retention vary by acquisition channel (e.g. tracker_name)?
- Are users acquired via paid campaigns more likely to generate ad revenue?

### 🔗 Correlations
- **Retention vs. install date**: Are certain days of the week or months associated with higher retention?
- **Retention vs. platform**: Does Android vs. iOS show different retention curves?
- **Retention vs. ad revenue**: Do retained users contribute more to monetization?

---

## 💰 Marketing Efficiency

### 🔍 Questions
- Which tracker yields the best ROI (revenue vs. cost)?
- How does CPI vary over time for each campaign?
- Are there diminishing returns for high-cost campaigns?

### 🔗 Correlations
- **CPI vs. installs**: Is lower CPI associated with higher install volume?
- **Campaign cost vs. retention**: Do expensive campaigns yield more loyal users?
- **Ad revenue vs. tracker**: Which campaigns drive the most monetizable users?

---

## 🌍 Geographic & Platform Insights

### 🔍 Questions
- Which countries have the highest install-to-retention conversion?
- Is ad revenue per user higher in certain regions?
- Do platform-specific behaviors differ in session depth or frequency?

### 🔗 Correlations
- **Country vs. ad revenue**: Are some markets more lucrative?
- **Platform vs. CPI**: Is it cheaper to acquire users on Android vs. iOS?
- **Country vs. retention**: Are cultural or regional patterns observable?

---

## 🧠 Advanced Causal Inference (if time-series or user-level granularity is available)

### 🔍 Questions
- Does early engagement (e.g. first 24h activity) predict long-term retention?
- Can we identify churn predictors based on event sequences?
- What is the causal impact of tracker cost on downstream revenue?

### 🔗 Techniques
- **Propensity score matching**: Compare similar users across campaigns.
- **Survival analysis**: Model time-to-churn.
- **Granger causality**: Test if one time series (e.g. installs) predicts another (e.g. revenue).

---